**Vision-Based Gaze Estimation | Presented by: Reza Saadatyar (2026)**

**03 | Classical Gaze Estimation Baselines**

**Objective:** Compare a training-derived mean-gaze reference against HOG descriptors and regularised Ridge regression for **unseen MPIIGaze participants**.

**Data:** Notebook 02's selected normalised eye images, 3D unit gaze vectors and nested participant folds. This is a study-defined bounded source subset rather than the official MPIIGaze benchmark split.

**Scientific boundary:** Ridge regularisation is selected solely on inner Validation people in each outer fold; the outer Test person is never used for fitting or tuning. This notebook saves descriptive outer-fold predictions and participant-level angular errors.

**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess
repo_url="https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path=Path("/content")/repo_url.rsplit("/",1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git","clone","-q",repo_url,str(repo_path)],check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}

**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root=setup_colab("vision_based_gaze_estimation")

# ---------------------------- 2. Standard Library ----------------------------
import sys

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------ 4. Classical Modelling and Evaluation --------------------
from gaze_models import hog_descriptors, normalise_directions
from gaze_models import fit_ridge_and_predict
from gaze_geometry import angular_error_degrees
from gaze_evaluation import load_image_cache, outer_fold_indices
from gaze_evaluation import person_angular_metrics

# ------------------- 5. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

# --------------------- 6. Output Directory and Setup -------------------------
out_dir=root/"vision_based_gaze_estimation"/"output"
cache_path=root/"data"/"processed"/"mpiigaze"/"02_normalised_eye_cache.npz"
plot_style(font_family="DejaVu Sans", title_size=12.5,
 title_weight="normal",label_size=11.5,label_weight="normal",
 tick_size=11,legend_size=10,figure_dpi=100,figure_size=(9,4))
print("Output directory:",out_dir)
print("Image cache:",cache_path)
print("Python:",sys.version.split()[0])

**02. Check Image–Label Identity and Fixed Fold Roles**

The sample index and local image cache originate from Notebook 02. They must agree on sample order and 3D vectors. A missing cache requires regenerating Notebook 02, not silently substituting an old or unlabelled image file.

In [ ]:
# ================ 02. Verify Source-Aligned Modelling Inputs =================

required=["02_sample_index.csv","02_fold_assignments.csv"]
missing=[x for x in required if not (out_dir/x).is_file()]
if missing:
    raise FileNotFoundError(f"Notebook 02 inputs missing: {missing}")
index,images,targets=load_image_cache(
 out_dir/"02_sample_index.csv",cache_path)
folds=pd.read_csv(out_dir/"02_fold_assignments.csv",
 dtype={"participant_id":str,"outer_test_person":str,"fold_id":str})
if folds.empty or folds.duplicated(["fold_id","participant_id"]).any():
    raise ValueError("Missing or repeated outer-fold participant assignments")
print("Source-aligned images:",len(index))
print("Independent people:",index.participant_id.nunique())
print("Fixed outer folds:",folds.fold_id.nunique())

**03. Compute Classical Eye-Image Descriptors**

HOG features are computed from image pixels only, independently of the gaze label. The same descriptor dimensions apply to every person. Mean-gaze uses only 3D direction vectors from each fold's fitting people.

In [ ]:
# ================== 03. Compute Source-Only HOG Descriptors ==================

features=hog_descriptors(images)
if len(features)!=len(index) or not np.isfinite(features).all():
    raise ValueError("HOG arrays do not match input sample identity")
print("HOG feature matrix:",features.shape)
print("No target or participant ID enters the image descriptor")

**04. Fit and Select Models Within Each Outer Fold**

The mean-gaze baseline is fitted only on the fold's training participants. Ridge hyperparameter selection compares training-fitted models on different **inner Validation people**. Final outer-Test metrics are descriptive estimates for the predefined procedures, not a tuning criterion.

In [ ]:
# ==================== 04. Fit Nested Classical Baselines =====================

# -------------------------- 1. Candidate Design ------------------------------
SEED=17
RIDGE_ALPHAS=(1.0,10.0,100.0)
outer_rows=[]
inner_rows=[]

# ---------------------- 2. Train and Select Within Fold ----------------------
for fold_id in sorted(folds.fold_id.unique()):
    role=outer_fold_indices(index,folds,fold_id)
    i_train,i_val,i_test=(role[k] for k in ("train","validation","test"))
    train_mean=normalise_directions(targets[i_train].mean(axis=0)[None])[0]
    prior=np.tile(train_mean,(len(i_test),1))
    best_alpha,best_error=None,float("inf")
    for alpha in RIDGE_ALPHAS:
        candidate,pred_val=fit_ridge_and_predict(
            features[i_train],targets[i_train],features[i_val],alpha)
        error=float(angular_error_degrees(targets[i_val],pred_val).mean())
        inner_rows.append({"fold_id":fold_id,"model":"hog_ridge",
            "alpha":alpha,"inner_val_mean_angular_error_deg":error,
            "n_inner_people":index.iloc[i_val].participant_id.nunique()})
        if error<best_error:
            best_alpha,best_error=alpha,error
    selected,pred_ridge=fit_ridge_and_predict(
        features[i_train],targets[i_train],features[i_test],best_alpha)
    for name,prediction in (("mean_direction",prior),("hog_ridge",pred_ridge)):
        rows=index.iloc[i_test][["sample_id","participant_id"]].copy()
        rows["fold_id"]=fold_id
        rows["model"]=name
        rows[["target_x","target_y","target_z"]]=targets[i_test]
        rows[["pred_x","pred_y","pred_z"]]=prediction
        rows["selected_inner_alpha"]=(best_alpha if name=="hog_ridge" else np.nan)
        outer_rows.append(rows)
    print(fold_id,"Test person:",index.iloc[i_test].participant_id.unique()[0],
          "| inner-selected Ridge alpha:",best_alpha)

outer_predictions=pd.concat(outer_rows,ignore_index=True)
inner_validation=pd.DataFrame(inner_rows)
print("Outer-fold prediction rows:",len(outer_predictions))

**05. Inspect Participant-Level Angular Error**

Angular errors are calculated between 3D direction vectors in the same normalised coordinate frame. Each outer fold contributes one independent Test person; multiple image predictions from that person are correlated.

In [ ]:
# ================== 05. Summarise Independent-Person Errors ==================

recorded,per_person=person_angular_metrics(outer_predictions)
if per_person.groupby("model").participant_id.nunique().min()!=index.participant_id.nunique():
    raise ValueError("Not all outer Test people have matched predictions")
summary=per_person.groupby("model").agg(
 n_people=("participant_id","nunique"),
 mean_person_angular_error_deg=("mean_angular_error_deg","mean"),
 median_person_angular_error_deg=("mean_angular_error_deg","median"),
 worst_person_angular_error_deg=("mean_angular_error_deg","max"))
display(summary)
print("No model was selected using outer-Test scores")

**06. Save Predictions, Metrics and Figures**

Save the stage-owned outer predictions and inner selection record. Do not use Test performance to change the model or parameter grid after seeing these results. The figure shows **per-participant** mean angular error, not overall image-wise accuracy.

In [ ]:
# =================== 06. Save Classical Modelling Evidence ===================

outer_predictions.to_csv(
 out_dir/"03_baseline_outer_predictions.csv",index=False)
per_person.to_csv(out_dir/"03_baseline_fold_metrics.csv",index=False)
inner_validation.to_csv(out_dir/"03_inner_validation_metrics.csv",index=False)
fig,ax=plt.subplots(figsize=(10,4.5))
plot_table=per_person.pivot(index="participant_id",columns="model",
 values="mean_angular_error_deg").sort_index()
plot_table.plot(kind="bar",ax=ax)
ax.set(title="Classical gaze estimation: held-out person errors",
 xlabel="Outer Test person",ylabel="Mean angular error (degrees)")
ax.grid(axis="y",alpha=0.2)
fig.tight_layout()
save_figure(fig,out_dir/"03_baseline_comparison.png")
plt.show()

**07. Scientific Interpretation and Output Download**

Results describe the source-selected image subset and the implemented nested person-level protocol. Whether HOG improves over the training-direction prior is an empirical question; this notebook makes no prior claim about the ordering.

In [ ]:
# ================ 07. Interpret Classical Results and Export =================

needed=["03_baseline_outer_predictions.csv",
 "03_baseline_fold_metrics.csv","03_inner_validation_metrics.csv",
 "03_baseline_comparison.png"]
missing=[x for x in needed if not (out_dir/x).is_file()]
if missing:
    raise FileNotFoundError(f"Stage-03 outputs missing: {missing}")
print("Outer-fold people evaluated:",per_person.participant_id.nunique())
print("Lower angular error is better; do not retune from outer-Test results")
print("Next: 04_cnn_and_transfer_learning.ipynb")
download_outputs_from_colab("vision_based_gaze_estimation/output","03")